# CLM-0.4-mini M1 Calibration

**Development calibration only.** This notebook opens seed `90401` and must never run formal seeds `90411/90412/90413`. It selects the first passing optimizer configuration in the pre-registered order; it does not emit an M1 scientific decision.

The performance path batches structural validation, groups equal Cell routes, caches direct-only baselines, uses CUDA AMP for training only, and uses both visible T4 GPUs when available. Validation remains FP32.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO_URL = 'https://github.com/ArcheLabs/mini-cells.git'
REPO_REF = 'main'
ROOT = Path('/kaggle/working/mini-cells')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',REPO_REF,REPO_URL,str(ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',REPO_REF,'--depth','1'], check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout',REPO_REF], check=True)
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin',REPO_REF], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-e','.[dev,lm]','-q'], cwd=ROOT, check=True)
subprocess.run(['git','-C',str(ROOT),'status','--short'], check=True)
print('Repository:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'], text=True).strip())


## Restore deterministic data assets after a Kaggle session reset

`/kaggle/working` is ephemeral. If the previous session was stopped, the frozen 30M-token data directory is expected to be gone. The next cell automatically rebuilds it from the already frozen TinyStories revision, then verifies every registered hash before calibration. This reconstruction is seed-independent and does **not** observe `90401`.

Calibration outputs under `/kaggle/working/clm-0.4-mini-calibration` are also ephemeral. If a full Kaggle session is stopped mid-calibration, the base checkpoint and candidate cache disappear too; rerunning this notebook will reconstruct data and restart from the beginning unless you separately persist that output as a Kaggle Dataset.

In [ ]:
DATA = Path('/kaggle/working/clm-0.4-mini-data')
OUT = Path('/kaggle/working/clm-0.4-mini-calibration')
PINNED_REVISION = 'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64'
EXPECTED_PATH = ROOT/'research/validations/clm-0.4-mini-language-validation/calibration-assets.json'
expected = json.loads(EXPECTED_PATH.read_text())

if not (DATA/'asset-summary.json').is_file():
    print('Frozen data assets are absent; rebuilding them for this Kaggle session...')
    subprocess.run([
        sys.executable, str(ROOT/'scripts/research/prepare_clm_0_4_mini_data.py'),
        '--dataset-revision', PINNED_REVISION,
        '--routing-salt', 'clm-0.4-mini-v1',
        '--out', str(DATA),
    ], cwd=ROOT, check=True)

assets = json.loads((DATA/'asset-summary.json').read_text())
checks = {
    'base_tokens': assets['base_tokens'],
    'tokenizer_hash': assets['tokenizer_hash'],
    'tokenizer_manifest_hash': assets['tokenizer_manifest_hash'],
    'base_corpus_manifest_hash': assets['base_corpus_manifest_hash'],
    'curriculum_manifest_hash': assets['curriculum_manifest_hash'],
    'routing_salt': assets['routing_salt'],
}
for key, value in checks.items():
    assert value == expected[key], f'{key}: expected {expected[key]!r}, got {value!r}'
assert assets['development_seed_observed'] is False
assert assets['formal_seeds_observed'] is False
print(json.dumps(checks, indent=2))


In [ ]:
# Validate the pre-registered plan without observing development seed 90401.
PLAN_OUT = Path('/kaggle/working/clm-0.4-mini-calibration-plan-check')
subprocess.run([sys.executable, str(ROOT/'scripts/research/run.py'), 'clm-0.4-mini-calibration', '--plan-only', '--out', str(PLAN_OUT)], cwd=ROOT, check=True)
plan = json.loads((PLAN_OUT/'calibration-plan.json').read_text())
assert plan['candidate_count'] == 81
plan['plan_sha256'], plan['candidates'][0], plan['candidates'][-1]


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before calibration.'
GPU_COUNT = torch.cuda.device_count()
CUDA_DEVICES = ','.join(f'cuda:{i}' for i in range(GPU_COUNT))
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
print('GPUs:', GPU_COUNT, GPU_NAMES)
print('Calibration devices:', CUDA_DEVICES)
GPU_COUNT, GPU_NAMES, torch.__version__, torch.version.cuda


## Open development seed 90401

The next cell is the **first allowed observation of development seed `90401`**. It trains the ~5M base model once on the frozen 30M-token corpus, checks base prerequisites, then evaluates candidates in the committed order and stops at the first full development-gate pass.

With two visible T4s, base pretraining uses address-aware data parallelism. During candidate search, cached `local_always`/`local_tx` baselines run independently of the growth variant, so the first occurrence of a direct configuration can use both GPUs concurrently. Later candidates with the same direct configuration reuse those baselines.

Do not edit the grid, gates, asset hashes, curriculum, routing salt, candidate order, or precision policy after running this cell.

In [ ]:
subprocess.run([
    sys.executable, str(ROOT/'scripts/research/run.py'), 'clm-0.4-mini-calibration',
    '--data-dir', str(DATA), '--out', str(OUT), '--device', 'cuda',
    '--devices', CUDA_DEVICES,
    '--seed', '90401', '--confirm-development-seed', '90401',
], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(ROOT/'scripts/research/report.py'), 'clm-0.4-mini-calibration', '--results', str(OUT)], cwd=ROOT, check=True)


In [ ]:
decision = json.loads((OUT/'decision.json').read_text())
assert decision['scientific_decision'] is False
assert decision['development_seed_observed'] is True
assert decision['formal_seeds_observed'] is False
decision


In [ ]:
summary = json.loads((OUT/'summary.json').read_text())
print('Execution engine:', summary.get('performance_format'))
print('Environment:', json.dumps(summary.get('environment', {}), indent=2))
if (OUT/'calibration-summary.csv').is_file():
    print('Candidate summary:', OUT/'calibration-summary.csv')

if decision['status'] == 'CALIBRATION_CONFIGURATION_SELECTED':
    selected = json.loads((OUT/'selected.json').read_text())
    lock = json.loads((OUT/'protocol-lock.candidate.json').read_text())
    print(json.dumps(selected['candidate'], indent=2))
    print('Protocol lock candidate:', OUT/'protocol-lock.candidate.json')
else:
    print('No formal seed may be opened. Status:', decision['status'])


## Next boundary

If and only if calibration selects a configuration, review `protocol-lock.candidate.json` and commit it as the canonical `research/validations/clm-0.4-mini-language-validation/protocol-lock.json` in a separate lock step. **Formal seeds remain forbidden until that commit exists.**